### 安装
- 安装python(>=3.12)和git
- pip install quarkstudio
- quark init

### 参数表

In [13]:
Q0 = {  # qubit节点
    "Measure": {  # 生成读取波形，送入probe通道
        "duration": 4e-06,
        "amp": 0.019,
        "frequency": 6964370000.0,
        "weight": "const(1)",
        "phi": -2.636421695283167,
        "threshold": 8502633802.265065,
        "ring_up_amp": 0.024,
        "ring_up_waist": 0.006,
        "ring_up_time": 6e-07
    },
    "acquire": {  # 采集通道（从制冷机出来的信号）
        "address": "AD.CH13.IQ",
        "TRIGD": 0
    },
    "probe": {  # 读取通道（进入制冷机的信号）
        "address": "AWG.CH2.Waveform",
        "delay": 0
    }

}


staion = { # 一些全局设置
    "sample": "test_111",
    "triggercmds": [
        "Trigger.CH1.TRIG"  # 触发设备
    ]
}


dev = {  # 设备列表
    'AD': {  # 采集卡
        "addr": "192.168.1.2",  # 设备ip
        "name": "dev.VirtualDevice",  # 设备模块路径，换成实际设备驱动
        "type": "driver"  # 直连设备，即无需通过remote服务
    },
    'AWG': {  # 任意波形发生器
        "addr": "192.168.1.3",
        "name": "dev.VirtualDevice",
        "type": "driver"
    },
    'Trigger': {  # 触发源
        "addr": "192.168.1.4",
        "name": "dev.VirtualDevice",
        "type": "driver"
    }
}

### 注册登录

In [4]:
from quark.app import s, Recipe
import numpy as np

Recipe.lib = 'lib.gates.u3rcp'  # 当前notebook范围内有效
Recipe.arch = 'rcp'  # 当前notebook范围内有效

- **注意：以下步骤要求server已经成功启动**

In [5]:
# 如果login报错，提示 LookupError: User "test" not found, signup first!
# 则运行下一行signup，成功后再回到这里login
s.login('test')

2025-07-31 13:56:21.165 | INFO     | quark.app:login:171 - LOGINED[test], Checkpoint[2025-07-31 13:44:31 Thu] loaded from: C:\Users\drice\Desktop\home\cfg\myexperiment.ckpt


In [4]:
# 注册用户test，并将比特参数存于checkpoint.json（默认位于~/Desktop/home/cfg）
# siginup执行一次就好
# s.signup('test','myexperiment') 

- **添加比特和设备到server**
    > 添加完后注意不要反复运行，否则会覆盖之前的

In [ ]:
# 添加触发设置
s.update('station', staion)  # 'Q0'名称任意

# 添加比特
s.update('Q0', Q0)  # 'Q0'名称任意

# 添加设备
for k, v in dev.items():
    s.update(f'dev.{k}', v)

In [8]:
# 打开设备，必须运行并确保设备全部正常打开
s.start()

'QuarkServer started!'

- 如果设备打开异常，直接导入驱动快速测试，定位问题所在

In [ ]:
# 根据设备类型导入实际设备驱动
from dev import VirtualDevice

# 根据设备地址实例化设备
d = VirtualDevice.Driver('192.168.1.42')
# 打开设备，多数错误出在一步
d.open()

# 设备写操作，通过setValue（实际调用`write`方法），可操作属性见驱动文件定义的quants列表
d.setValue('Power', -10)
# 设备读操作，通过getValue（实际调用`read`方法），可操作属性见驱动文件定义的quants列表
d.getValue('Power')

-10

### AD S21

In [12]:
def S21(qubits: tuple[str], freq: float, ctx=None) -> list:
    """qlisp线路函数。ctx为编译所需上下文，主要用于对cfg表进行查询等操作。
    """
    cc = [(('Measure', i), q) for i, q in enumerate(qubits)]
    return cc


rcp = Recipe('s21', signal='iq_avg')
rcp.circuit = S21  # 指定扫描线路函数

qubits = ['Q0']
rcp['qubits'] = tuple(qubits)  # 必须为tuple
rcp['freq'] = np.linspace(-10, 10, 101) * 1e6  # 扫描范围

for q in qubits:
    rcp[f'{q}.Measure.frequency'] = rcp['freq'] + \
        s.query(f'{q}.Measure.frequency')  # 在中心频率正负10M范围内扫描

s21 = s.submit(rcp.export(),
               block=False,  # 是否阻塞当前任务至结束
               preview=['Q0'],  # 需要打开quark canvas
               plot=True  # 需要打开quark viewer
               )
s21.bar()

baqis:/s21(tid=2507311359248004560)   0%|          |0/101 [00:00<?, ?it/s, MainThread]

### NA S21

In [50]:
def circuit(power: float, flux: float, ctx=None):
    cc = [(('SET', 'FrequencyStart', 6.85 * 1e9), 'NA.CH1'),  # 可设置属性，见网分驱动
          (('SET', 'FrequencyStop', 7e9), 'NA.CH1'),
          (('SET', 'NumberOfPoints', 5001), 'NA.CH1'),
          (('SET', 'Power', power), 'NA.CH1'),
          (('setBias', 'flux', flux), 'Q0'),  # 编译生成偏置波形，非Offset
          (('GET', 'S'), 'NA.CH1')
          ]
    return cc


rcp = Recipe('NAS21', signal='S')
rcp.circuit = circuit

rcp['power'] = np.linspace(-60, 0, 3)  # 扫描power
rcp['flux'] = np.linspace(-1, 1, 2)  # 扫描flux


tt = s.submit(rcp.export())
tt.bar()

baqis:/NAS21(tid=2507311343038964503)   0%|          |0/6 [00:00<?, ?it/s, MainThread]